# AF2DECAY1 — late auxiliary release
One seed, 30 epochs, cosine gain to zero over final 10 epochs. No test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json,os,shutil,subprocess,sys,tarfile,time
from pathlib import Path
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/af2-signal-preservation-deep-supervision'; os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result=subprocess.run(clone,cwd=WORK)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('Git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact,resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
ARCHIVE_REL='bundles/faruq-development-v3-grouped.tar'; AF2_REL='experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'
PROJECT=resolve_drive_project_root(required_relative_paths=(ARCHIVE_REL,AF2_REL)); ARCHIVE=require_project_artifact(PROJECT,ARCHIVE_REL); AF2=require_project_artifact(PROJECT,AF2_REL)
DATA=WORK/'faruq-development-v3-grouped'
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall(WORK,filter='data')
assert (DATA/'data.yaml').is_file() and not (DATA/'test').exists(); GROUPED=DATA/'faruq_grouped_summary.json'; assert GROUPED.is_file()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-spds-refinement-v1'; STATIC=OUTPUT/'static_audit.json'; assert STATIC.is_file(),'Jalankan static audit refinement.'
print('GPU:',torch.cuda.get_device_name(0)); print('OUTPUT:',OUTPUT)

In [ ]:
ARM='AF2DECAY1'; LOG=OUTPUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_spds_refinement_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    last=-1
    while process.poll() is None:
        csv=OUTPUT/ARM/f'{ARM}_seed42'/'results.csv'; epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epochs!=last and (epochs==0 or epochs%5==0): print(f'{ARM}: {epochs}/30 epoch tercatat',flush=True)
        last=epochs; time.sleep(30)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-50:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
print('SELESAI:',ARM)

In [ ]:
RESULT=OUTPUT/'val_reports'/'AF2DECAY1_seed42_result.json'; payload=json.loads(RESULT.read_text())
print({key:payload['metrics'][key] for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}); print('TEST:',payload['test_images_accessed'])